adapted from https://github.com/LSSTDESC/transient-host-sims/blob/main/notebooks/SCOTCH_walkthroughs.ipynb
and https://github.com/LSSTDESC/transient-host-sims/blob/main/notebooks/pzflow_DC2_conditionalFlow_finalCut.ipynb
and https://jfcrenshaw.github.io/pzflow/tutorials/intro/

This notebook returns a pzflow model for SN Ia x1 and c distribution before selection effects based on the ZTF SN DR2 data. 
1. Train pzflow flow model using the x1, c and hostmass given in ZTF SN DR2.
2. Draw 100,000 random x1, c to obtain pdfs (we tested that x1, c are uncorrelated so we don't need a joint distribution).
3. Inverse-apply selection function to the pdfs in step 3. (pdf_before_selection=pdf_after_selection/selection_func) .
4. Draw 100,000 random x1, c from pdf_before_selection
5. Train another pzflow model conditioned on x1, c in step 4 (joint with hostmass).

In [ ]:
# import jax
# jax.default_backend()
# import os
# os.environ["JAX_PLATFORM_NAME"] = "cpu"

In [ ]:
import pandas as pd
import numpy as np
import pzflow
from pzflow import Flow
from pzflow.bijectors import Chain, ColorTransform, InvSoftplus, StandardScaler, RollingSplineCoupling
# from pzflow.examples import galaxy_data
from pzflow.distributions import Uniform, Joint, Normal
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
from scipy.stats.sampling import NumericalInversePolynomial
from scipy.stats import gaussian_kde

In [ ]:
n_sample = 100000

In [ ]:
globalhostdata = pd.read_csv('ztfsniadr2/tables/globalhost_data.csv')
localhostdata = pd.read_csv('ztfsniadr2/tables/localhost_data.csv')
sndata = pd.read_csv('ztfsniadr2/tables/snia_data.csv')

In [ ]:
globalhostdata.head()

In [ ]:
localhostdata.head()

In [ ]:
sndata.head()

In [ ]:
data = pd.merge(sndata,globalhostdata,on='ztfname')
data.head()

In [ ]:
data.columns

In [ ]:
data_train = data.loc[data.lccoverage_flag == 1,['mass','x1','c']]
data_train

In [ ]:
data_train = data_train.dropna()
data_train

In [ ]:
flow = Flow(data_train.columns)

In [ ]:
losses = flow.train(data_train, verbose=True)

In [ ]:
plt.plot(losses)
plt.xlabel("Epoch")
plt.ylabel("Training loss")
plt.show()

In [ ]:
flow.save("data/ztfsniadr2_host_sn_pzflow.pkl")

In [ ]:
flow = Flow(file="data/ztfsniadr2_host_sn_pzflow.pkl")

In [ ]:
samples = flow.sample(n_sample)

In [ ]:
samples.x1.hist(bins=20,alpha=0.3,density=True)
data_train.x1.hist(bins=20,alpha=0.3,density=True)

In [ ]:
samples.c.hist(bins=20,alpha=0.3,density=True)
data_train.c.hist(bins=20,alpha=0.3,density=True)

In [ ]:
plt.scatter(samples['x1'],samples['c'])

In [ ]:
cov = np.cov(samples['x1'],samples['c'])

In [ ]:
cov

In [ ]:
def expo(x, a, b, c):
    x = np.asarray(x)
    y = a * np.exp(b * x) + c
    return y

In [ ]:
# fx1_arr = np.loadtxt('data/ztf_selection_func_x1.txt')
# fc_arr = np.loadtxt('data/ztf_selection_func_c.txt')
fx1_parr = np.loadtxt('data/ztf_selection_func_x1.txt')
fc_parr = np.loadtxt('data/ztf_selection_func_c.txt')

In [ ]:
fx1_func = lambda x1: expo(x1, *fx1_parr)
fc_func = lambda c: expo(c, *fc_parr)

In [ ]:
bins_x1 = np.linspace(-5, 5, 30)
bins_c = np.linspace(-0.5, 1, 30) 

binwidth_x1 = bins_x1[1]-bins_x1[0]
binwidth_c = bins_c[1]-bins_c[0]

x1_binned_pdf,x1_bin_edges = np.histogram(samples['x1'],bins=bins_x1)
c_binned_pdf,c_bin_edges = np.histogram(samples['c'],bins=bins_c)

x1_bin_center = (x1_bin_edges[1:] + x1_bin_edges[:-1])*0.5 
c_bin_center = (c_bin_edges[1:] + c_bin_edges[:-1])*0.5 

x1_pdf_orig = gaussian_kde(samples['x1'])
c_pdf_orig = gaussian_kde(samples['c'])

x1_pdf_modified = lambda x1: x1_pdf_orig(x1)/fx1_func(x1)
c_pdf_modified = lambda c: c_pdf_orig(c)/fc_func(c)

plt.plot(x1_bin_center, fx1_func(x1_bin_center))
plt.show()

plt.plot(c_bin_center, fc_func(c_bin_center))
plt.show()

In [ ]:
# binwidth_x1 = fx1_arr[0][1]-fx1_arr[0][0]
# binwidth_c = fc_arr[0][1]-fc_arr[0][0]

# bins_x1 = np.append((fx1_arr[0]-0.5*binwidth_x1), fx1_arr[0][-1]+0.5*binwidth_x1)
# bins_c = np.append((fc_arr[0]-0.5*binwidth_c), fc_arr[0][-1]+0.5*binwidth_c)

# x1_binned_pdf,x1_bin_edges = np.histogram(samples['x1'],bins=bins_x1)
# c_binned_pdf,c_bin_edges = np.histogram(samples['c'],bins=bins_c)

# x1_binned_pdf_modified = x1_binned_pdf/fx1_arr[1]
# c_binned_pdf_modified = c_binned_pdf/fc_arr[1]

In [ ]:
plt.bar(x1_bin_center,x1_pdf_orig(x1_bin_center)/np.sum(x1_pdf_orig(x1_bin_center))/binwidth_x1,alpha=0.3,width=binwidth_x1)
plt.bar(x1_bin_center,x1_pdf_modified(x1_bin_center)/np.sum(x1_pdf_modified(x1_bin_center))/binwidth_x1,alpha=0.3,width=binwidth_x1)

In [ ]:
plt.bar(c_bin_center,c_pdf_orig(c_bin_center)/np.sum(c_pdf_orig(c_bin_center))/binwidth_c,alpha=0.3,width=binwidth_c)
plt.bar(c_bin_center,c_pdf_modified(c_bin_center)/np.sum(c_pdf_modified(c_bin_center))/binwidth_c,alpha=0.3,width=binwidth_c)

In [ ]:
x1_pdf = interp1d(x1_bin_center,x1_pdf_modified(x1_bin_center)/np.sum(x1_pdf_modified(x1_bin_center))/binwidth_x1,fill_value=[0.],bounds_error=False)
c_pdf = interp1d(c_bin_center,c_pdf_modified(c_bin_center)/np.sum(c_pdf_modified(c_bin_center))/binwidth_c,fill_value=[0.],bounds_error=False)

In [ ]:
# plt.bar(0.5*(c_bin_edges[:-1]+c_bin_edges[1:]),c_binned_pdf/np.sum(c_binned_pdf)/binwidth_c,alpha=0.3,width=binwidth_c)
# plt.bar(0.5*(c_bin_edges[:-1]+c_bin_edges[1:]),c_binned_pdf_modified/np.sum(c_binned_pdf_modified)/binwidth_c,alpha=0.3,width=binwidth_c)

In [ ]:
# x1_pdf = interp1d(fx1_arr[0],x1_binned_pdf_modified,fill_value=[0.],bounds_error=False)
# c_pdf = interp1d(fc_arr[0],c_binned_pdf_modified,fill_value=[0.],bounds_error=False)

In [ ]:
class x1distr:
   def pdf(self, x):
       return x1_pdf(x)

In [ ]:
class cdistr:
   def pdf(self, x):
       return c_pdf(x)

In [ ]:
urng = np.random.default_rng()

dist = x1distr()
rng = NumericalInversePolynomial(dist, random_state=urng)
x1 = rng.rvs(n_sample)

dist = cdistr()
rng = NumericalInversePolynomial(dist, random_state=urng)
c = rng.rvs(n_sample)

In [ ]:
flow_cond = Flow(data_columns=["mass"], conditional_columns=["x1","c"])
losses = flow_cond.train(data_train, verbose=True)
plt.plot(losses)
plt.xlabel("Epoch")
plt.ylabel("Training loss")
plt.show()

In [ ]:
df = pd.DataFrame({"x1":x1,"c":c})
samples_cond = flow_cond.sample(1, conditions=df)

In [ ]:
samples_cond.x1.hist(bins=20,alpha=0.3,density=True)
data_train.x1.hist(bins=20,alpha=0.3,density=True)

In [ ]:
samples_cond.c.hist(bins=20,alpha=0.3,density=True)
data_train.c.hist(bins=20,alpha=0.3,density=True)

In [ ]:
flow_before_selection = Flow(samples_cond.columns)
losses = flow_before_selection.train(samples_cond, verbose=True)
plt.plot(losses)
plt.xlabel("Epoch")
plt.ylabel("Training loss")
plt.show()

In [ ]:
flow_before_selection.save("data/ztfsniadr2_host_sn_before_selection_pzflow.pkl")

In [ ]:
flow = Flow(file="data/ztfsniadr2_host_sn_before_selection_pzflow.pkl")

In [ ]:
samples = flow.sample(1000)

In [ ]:
samples.x1.hist(bins=20,alpha=0.3,density=True)
data_train.x1.hist(bins=20,alpha=0.3,density=True)

In [ ]:
samples.c.hist(bins=20,alpha=0.3,density=True)
data_train.c.hist(bins=20,alpha=0.3,density=True)